El threading permite crear distintos hilos dentro de un mismo núcleo. Esto permite que nuestro programa paralelice tareas mientras comparte recursos (variables, ficheros, etc.).

En Python, existen dos librerías principales para esta tarea: threading y asyncio. La segunda es más extensa, utilizaremos la primera a modo introductorio en el mundo de la concurrencia.

In [1]:
import threading
import time

__Problema inicial__: tenemos una tarea exhaustiva que queremos acelerar.

In [2]:
%%time
 # IPython magic: con esto, se nos devuelve el tiempo que ha tardado la celda en ejecutarse
def tarea(numero):
    print(f"Iniciando tarea {numero}")
    time.sleep(2)  # Simula una tarea que tarda 2 segundos
    print(f"Tarea {numero} completada")

# Bucle secuencial
for i in range(5):
    tarea(i)

Iniciando tarea 0
Tarea 0 completada
Iniciando tarea 1
Tarea 1 completada
Iniciando tarea 2
Tarea 2 completada
Iniciando tarea 3
Tarea 3 completada
Iniciando tarea 4
Tarea 4 completada
CPU times: total: 0 ns
Wall time: 10 s


__Creación de hilos__: 
- Podemos heredar de la clase threading.Thread. La lógica que se debe realizar se declara en la función run() que debemos sobrescribir. Si hay que pasar parámetros, los pasamos mediante el constructor.
- Podemos crear directamente un hilo con la clase Thread, sin crear una nueva clase. En tal caso, hay que pasar al constructor la función a ejecutar (argumento *target*) y los argumentos de dicha función (argumento *args*).

In [4]:
%%time
class MiHilo(threading.Thread):
    def __init__(self, numero):
        super().__init__()
        self.numero = numero

    def run(self):
        print(f"Iniciando tarea {self.numero}")
        time.sleep(2)
        print(f"Tarea {self.numero} completada")

# Crear y lanzar los hilos
hilos = []
for i in range(5):
    hilo = MiHilo(i)
    # Nos listamos todos los hilos
    hilos.append(hilo)
    hilo.start()

# Esperar a que todos los hilos terminen
# for hilo in hilos:
#     hilo.join()

Iniciando tarea 0
Iniciando tarea 1
Iniciando tarea 2
Iniciando tarea 3
Iniciando tarea 4
CPU times: total: 0 ns
Wall time: 4.08 ms


Tarea 0 completada
Tarea 1 completada
Tarea 2 completada
Tarea 3 completada
Tarea 4 completada


In [5]:
%%time
# Bucle paralelo con threading
hilos = []
for i in range(5):
    hilo = threading.Thread(target=tarea, args=(i,))
    hilos.append(hilo)
    hilo.start()

# Esperar a que todos los hilos terminen
for hilo in hilos:
    hilo.join()

Iniciando tarea 0
Iniciando tarea 1
Iniciando tarea 2
Iniciando tarea 3
Iniciando tarea 4
Tarea 0 completada
Tarea 2 completada
Tarea 3 completada
Tarea 1 completada
Tarea 4 completada
CPU times: total: 0 ns
Wall time: 2 s


# Acceso a secciones críticas

Los hilos comparten variables, por lo que hay que tener cuidado de que varios hilos no traten de modificarlas simultáneamente, ya que podría dar lugar a un funcionamiento no deseado. Pongamos un ejemplo con transacciones bancarias.

In [ ]:
class CuentaBancaria:
    def __init__(self):
        self.saldo = 0

    def depositar(self, cantidad):
        print(f"[Depositar] Depositando {cantidad}€")
        time.sleep(0.3)  # Simula retraso entre lectura y escritura
        self.saldo += cantidad
        print(f"[Depositar] Saldo actual: {self.saldo}€")

    def retirar(self, cantidad):
        print(f"[Retirar] Intentando retirar {cantidad}€")
        time.sleep(0.25)  # Simula retraso entre lectura y escritura
        if self.saldo < cantidad:
            print(f"[Retirar] Saldo insuficiente ({self.saldo}€).")
            return
        self.saldo -= cantidad
        print(f"[Retirar] Retiro exitoso. Saldo restante: {self.saldo}€")


In [12]:

cuenta = CuentaBancaria()

hilos = [threading.Thread(target=cuenta.depositar, args=(100,)),
        threading.Thread(target=cuenta.retirar, args=(50,))
        ]  

for h in hilos:
    h.start()

for h in hilos:
    h.join()


[Depositar] Depositando 100€
[Retirar] Intentando retirar 50€
[Depositar] Saldo actual: 100€
[Retirar] Retiro exitoso. Saldo restante: 50€


Para solventar este problema, se implementa un *Condition* de la clase *threading*. Principalmente, hay dos métodos que permiten la concurrencia:
- *acquire*: al llamar a esta función, solo uno de los hilos puede continuar. El resto seguirá esperando en una cola.
- *release*: se avisa de que ya no nos encontramos en una sección crítica. Otro hilo que hubiera llamado a *acquire* puede continuar.

In [17]:
class CuentaBancaria:
    def __init__(self):
        self.saldo = 0
        self.condition = threading.Condition()

    def depositar(self, cantidad):
        self.condition.acquire() # Ahora, reservamos nosotros 
        
        print(f"[Depositar] Depositando {cantidad}€")
        time.sleep(0.3)  # Simula retraso entre lectura y escritura
        self.saldo += cantidad
        print(f"[Depositar] Saldo actual: {self.saldo}€")
        
        # self.condition.release() # Liberamos el recurso

    def retirar(self, cantidad):
        self.condition.acquire() # Ahora, reservamos nosotros 
        
        print(f"[Retirar] Intentando retirar {cantidad}€")
        time.sleep(0.25)  # Simula retraso entre lectura y escritura
        if self.saldo < cantidad:
            print(f"[Retirar] Saldo insuficiente ({self.saldo}€).")
            return
        self.saldo -= cantidad
        print(f"[Retirar] Retiro exitoso. Saldo restante: {self.saldo}€")
        
        self.condition.release() # Liberamos el recurso

In [ ]:
cuenta = CuentaBancaria()

hilos = [threading.Thread(target=cuenta.depositar, args=(100,)),
        threading.Thread(target=cuenta.retirar, args=(50,)),
        threading.Thread(target=cuenta.retirar, args=(20,))
        ]  

for h in hilos:
    h.start()

for h in hilos:
    h.join()

[Depositar] Depositando 100€
[Depositar] Saldo actual: 100€


Esta lógica es similar al *open* y *close* de los ficheros. De hecho, podemos usar el *with* igual:

In [29]:
class CuentaBancaria:
    def __init__(self):
        self.saldo = 0
        self.condition = threading.Condition()

    def depositar(self, cantidad):
        with self.condition:
            print(f"[Depositar] Depositando {cantidad}€")
            time.sleep(0.3)  # Simula retraso entre lectura y escritura
            self.saldo += cantidad
            print(f"[Depositar] Saldo actual: {self.saldo}€")

    def retirar(self, cantidad):
        with self.condition:
            print(f"[Retirar] Intentando retirar {cantidad}€")
            time.sleep(0.25)  # Simula retraso entre lectura y escritura
            if self.saldo < cantidad:
                print(f"[Retirar] Saldo insuficiente ({self.saldo}€).")
                return
            self.saldo -= cantidad
            print(f"[Retirar] Retiro exitoso. Saldo restante: {self.saldo}€")